## 🎛️ **SceneFake Audio Dataset**
- **Purpose:** Aims to tackle a **neglected threat scenario** in fake audio detection — manipulating the **acoustic background scene** of a real utterance while keeping the voice content intact.
  
- **Source:** Generated using **speech enhancement technologies** to forge or alter the background environment of existing real audios.

- **Preprocessing:**
  - Real utterances processed by replacing or enhancing the **acoustic scene**
  - Focused on scene manipulation without changing **timbre, prosody, or linguistic content**

---

## 🗣️ **Fake-or-Real Audio Dataset**
- **Size:** Over **170,000 utterances** from real human and computer-generated speech
 
- **Versions Available:**
  1. **for-original:** Raw, balanced files without modification
  2. **for-norm:** Balanced by **gender and class**, with normalized **sample rate, volume, and channels**
  3. **for-2sec:** Based on *for-norm*, but trimmed to **2-second clips**
  4. **for-rerec:** A **rerecorded version of for-2sec**, simulating playback through a voice channel (like a call or voice message)

In [1]:
import os
import pandas as pd
import numpy as np
import random
import subprocess

import torch
import torchaudio
import torchaudio.transforms as T
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

from transformers import ASTFeatureExtractor, ASTForAudioClassification, ASTConfig

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

from pydub import AudioSegment

2025-05-06 11:55:06.680653: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746532506.879172      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746532506.935616      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
def get_fake_real_data(root_dir):
    file_paths = []
    labels = []
    splits = []
    versions = []

    for version in os.listdir(root_dir):
        version_path = os.path.join(root_dir, version)
        if os.path.isdir(version_path):
            # Check if فيه فولدر واحد جواه (بغض النظر عن اسمه)
            subdirs = [d for d in os.listdir(version_path) if os.path.isdir(os.path.join(version_path, d))]

            if len(subdirs) == 1:
                target_path = os.path.join(version_path, subdirs[0])
            else:
                target_path = version_path

            for split in ['training', 'validation', 'testing']:
                split_path = os.path.join(target_path, split)
                if os.path.isdir(split_path):
                    for label in ['real', 'fake']:
                        label_path = os.path.join(split_path, label)
                        if os.path.isdir(label_path):
                            for file_name in os.listdir(label_path):
                                if file_name.lower().endswith(('.wav', '.mp3')):
                                    file_paths.append(os.path.join(label_path, file_name))
                                    labels.append(label)
                                    splits.append(split)
                                    versions.append(version)

    df = pd.DataFrame({
        'file_path': file_paths,
        'label': labels,
        'split': splits,
        'version': versions,
        'dataset': 'fake-or-real'
    })

    return df

fake_real_df = get_fake_real_data("/kaggle/input/the-fake-or-real-dataset")

fake_real_df

,file_path,label,split,version,dataset
0,/kaggle/input/the-fake-or-real-dataset/for-nor...,real,training,for-norm,fake-or-real
1,/kaggle/input/the-fake-or-real-dataset/for-nor...,real,training,for-norm,fake-or-real
2,/kaggle/input/the-fake-or-real-dataset/for-nor...,real,training,for-norm,fake-or-real
3,/kaggle/input/the-fake-or-real-dataset/for-nor...,real,training,for-norm,fake-or-real
4,/kaggle/input/the-fake-or-real-dataset/for-nor...,real,training,for-norm,fake-or-real
...,...,...,...,...,...
169749,/kaggle/input/the-fake-or-real-dataset/for-rer...,fake,testing,for-rerec,fake-or-real
169750,/kaggle/input/the-fake-or-real-dataset/for-rer...,fake,testing,for-rerec,fake-or-real
169751,/kaggle/input/the-fake-or-real-dataset/for-rer...,fake,testing,for-rerec,fake-or-real
169752,/kaggle/input/the-fake-or-real-dataset/for-rer...,fake,testing,for-rerec,fake-or-real


In [3]:
# Group by split, version, and label
grouped_counts = fake_real_df.groupby(['split', 'version', 'label']).size().reset_index(name='count')


fig = px.bar(grouped_counts, 
             x='split', 
             y='count', 
             color='label', 
             facet_col='version',
             title='Number of Real and Fake Audios per Split and Version',
             barmode='group')

fig.show()

In [4]:
fake_real_df = fake_real_df.drop(columns=['version'])

In [5]:
def get_scenefake_data(root_dir):
    file_paths = []
    labels = []
    splits = []

    for split in ['train', 'dev', 'eval']:
        split_path = os.path.join(root_dir, split)
        if os.path.isdir(split_path):
            for label in ['real', 'fake']:
                label_path = os.path.join(split_path, label)
                if os.path.isdir(label_path):
                    for file_name in os.listdir(label_path):
                        if file_name.lower().endswith('.wav'):
                            file_paths.append(os.path.join(label_path, file_name))
                            labels.append(label)
                            splits.append(split)

    df = pd.DataFrame({
        'file_path': file_paths,
        'label': labels,
        'split': splits,
        'dataset': 'scenefake'
    })

    return df

scenefake_df = get_scenefake_data("/kaggle/input/scenefake")

scenefake_df

,file_path,label,split,dataset
0,/kaggle/input/scenefake/train/real/A_038_0_C.wav,real,train,scenefake
1,/kaggle/input/scenefake/train/real/A_2380_20_B...,real,train,scenefake
2,/kaggle/input/scenefake/train/real/A_309_0_A.wav,real,train,scenefake
3,/kaggle/input/scenefake/train/real/A_313_0_B.wav,real,train,scenefake
4,/kaggle/input/scenefake/train/real/A_2570_20_D...,real,train,scenefake
...,...,...,...,...
58769,/kaggle/input/scenefake/eval/fake/C_31220_10_D...,fake,eval,scenefake
58770,/kaggle/input/scenefake/eval/fake/C_21608_5_A.wav,fake,eval,scenefake
58771,/kaggle/input/scenefake/eval/fake/C_15804_20_A...,fake,eval,scenefake
58772,/kaggle/input/scenefake/eval/fake/C_23236_05_B...,fake,eval,scenefake


In [6]:
# Group by split and label
grouped_counts_scenefake = scenefake_df.groupby(['split', 'label']).size().reset_index(name='count')

fig = px.bar(grouped_counts_scenefake, 
             x='split', 
             y='count', 
             color='label',
             title='Number of Real and Fake Audios per Split in SceneFake Dataset',
             barmode='group')

fig.show()

In [7]:
fig = px.pie(scenefake_df, 
             names='label', 
             title='Label Distribution in SceneFake Dataset', 
             hole=0.5)  

fig.show()

In [8]:
scenefake_df = scenefake_df.drop(columns=['split'])

real_df = scenefake_df[scenefake_df['label'] == 'real']
fake_df = scenefake_df[scenefake_df['label'] == 'fake'].sample(n=len(real_df), random_state=42)

scenefake_df = pd.concat([real_df, fake_df]).sample(frac=1, random_state=42).reset_index(drop=True)

In [9]:
scenefake_df['label'] = scenefake_df['label'].replace({'fake': 'scene_fake', 'real': 'scene_real'})
fake_real_df['label'] = fake_real_df['label'].replace({'fake': 'voice_fake', 'real': 'voice_real'})

In [10]:
scenefake_train_df, temp_df = train_test_split(scenefake_df, test_size=0.2, random_state=42, stratify=scenefake_df['label'])

scenefake_val_df, scenefake_test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"Train size: {len(scenefake_train_df)}")
print(f"Validation size: {len(scenefake_val_df)}")
print(f"Test size: {len(scenefake_test_df)}")

Train size: 18251
Validation size: 2281
Test size: 2282


In [11]:
fake_real_train_df = fake_real_df[fake_real_df['split'] == 'training']

fake_real_train_df = fake_real_train_df.groupby('label').apply(lambda x: x.sample(8000, random_state=42)).reset_index(drop=True)

scenefake_train_df = scenefake_train_df.groupby('label').apply(lambda x: x.sample(8000, random_state=42)).reset_index(drop=True)

final_train_df = pd.concat([fake_real_train_df, scenefake_train_df]).reset_index(drop=True)

final_train_df = final_train_df.drop(columns=['split', 'dataset'])

final_train_df

/tmp/ipykernel_31/239155867.py:3: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_31/239155867.py:5: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,file_path,label
0,/kaggle/input/the-fake-or-real-dataset/for-ori...,voice_fake
1,/kaggle/input/the-fake-or-real-dataset/for-ori...,voice_fake
2,/kaggle/input/the-fake-or-real-dataset/for-ori...,voice_fake
3,/kaggle/input/the-fake-or-real-dataset/for-nor...,voice_fake
4,/kaggle/input/the-fake-or-real-dataset/for-2se...,voice_fake
...,...,...
31995,/kaggle/input/scenefake/train/real/A_2104_15_D...,scene_real
31996,/kaggle/input/scenefake/eval/real/C_2894_10_A.wav,scene_real
31997,/kaggle/input/scenefake/dev/real/B_1178_20_A.wav,scene_real
31998,/kaggle/input/scenefake/eval/real/C_5871_10_A.wav,scene_real


In [12]:
fake_real_val_df = fake_real_df[fake_real_df['split'] == 'validation']

fake_real_val_df = fake_real_val_df.groupby('label').apply(lambda x: x.sample(800, random_state=42)).reset_index(drop=True)

scenefake_val_df = scenefake_val_df.groupby('label').apply(lambda x: x.sample(800, random_state=42)).reset_index(drop=True)

final_val_df = pd.concat([fake_real_val_df, scenefake_val_df]).reset_index(drop=True)

final_val_df = final_val_df.drop(columns=['split', 'dataset'])

final_val_df

/tmp/ipykernel_31/73116454.py:3: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_31/73116454.py:5: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,file_path,label
0,/kaggle/input/the-fake-or-real-dataset/for-nor...,voice_fake
1,/kaggle/input/the-fake-or-real-dataset/for-ori...,voice_fake
2,/kaggle/input/the-fake-or-real-dataset/for-2se...,voice_fake
3,/kaggle/input/the-fake-or-real-dataset/for-nor...,voice_fake
4,/kaggle/input/the-fake-or-real-dataset/for-nor...,voice_fake
...,...,...
3195,/kaggle/input/scenefake/train/real/A_507_05_C.wav,scene_real
3196,/kaggle/input/scenefake/eval/real/C_4832_05_A.wav,scene_real
3197,/kaggle/input/scenefake/eval/real/C_5563_5_C.wav,scene_real
3198,/kaggle/input/scenefake/train/real/A_398_0_A.wav,scene_real


In [13]:
fake_real_test_df = fake_real_df[fake_real_df['split'] == 'testing']

fake_real_test_df = fake_real_test_df.groupby('label').apply(lambda x: x.sample(800, random_state=42)).reset_index(drop=True)

scenefake_test_df = scenefake_test_df.groupby('label').apply(lambda x: x.sample(800, random_state=42)).reset_index(drop=True)

final_test_df = pd.concat([fake_real_test_df, scenefake_test_df]).reset_index(drop=True)

final_test_df = final_test_df.drop(columns=['split', 'dataset'])

final_test_df

/tmp/ipykernel_31/963909442.py:3: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_31/963909442.py:5: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,file_path,label
0,/kaggle/input/the-fake-or-real-dataset/for-nor...,voice_fake
1,/kaggle/input/the-fake-or-real-dataset/for-nor...,voice_fake
2,/kaggle/input/the-fake-or-real-dataset/for-ori...,voice_fake
3,/kaggle/input/the-fake-or-real-dataset/for-nor...,voice_fake
4,/kaggle/input/the-fake-or-real-dataset/for-rer...,voice_fake
...,...,...
3195,/kaggle/input/scenefake/dev/real/B_1884_0_B.wav,scene_real
3196,/kaggle/input/scenefake/train/real/A_266_0_B.wav,scene_real
3197,/kaggle/input/scenefake/train/real/A_1298_10_C...,scene_real
3198,/kaggle/input/scenefake/dev/real/B_0087_05_A.wav,scene_real


In [14]:
# Set random seed for reproducibility
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Parameters
sample_rate = 16000  # AST expects 16kHz, feature extractor handles resampling
batch_size = 32
num_epochs = 10
learning_rate = 1e-4

# Load feature extractor
feature_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

# Load model with modified config for 4 labels
config = ASTConfig.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593", num_labels=4)
model = ASTForAudioClassification.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593", config=config, ignore_mismatched_sizes=True)
model.to(device)


# Freeze all parameters except last two transformer layers and classifier head
for name, param in model.named_parameters():
    param.requires_grad = False
for name, param in model.named_parameters():
    if 'ast.encoder.layer.10' in name or 'ast.encoder.layer.11' in name or 'classifier' in name:
        param.requires_grad = True

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
# Label encoding
le = LabelEncoder()
le.fit(final_train_df['label'])
final_train_df['label_enc'] = le.transform(final_train_df['label'])
final_val_df['label_enc'] = le.transform(final_val_df['label'])
final_test_df['label_enc'] = le.transform(final_test_df['label'])

# Optimizer and loss
optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [16]:
wav_dir = '/kaggle/working/wav_files/'
os.makedirs(wav_dir, exist_ok=True)

# Function to convert MP3 to WAV
def convert_mp3_to_wav(df, dataset_type):
    """
    Convert MP3 files to WAV, check all files with torchaudio.load, skip failed conversions and invalid files
    """
    new_paths = []
    valid_indices = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Converting MP3 to WAV for {dataset_type}"):
        audio_path = row['file_path']
        target_path = audio_path
        
        if audio_path.lower().endswith('.mp3'):
            try:
                # Load MP3 file
                audio = AudioSegment.from_mp3(audio_path)
                # Define new WAV path
                wav_path = os.path.join(wav_dir, f"{dataset_type}_{idx}.wav")
                # Export as WAV with 16kHz sampling rate
                audio = audio.set_frame_rate(16000)
                audio.export(wav_path, format="wav")
                target_path = wav_path
            except Exception as e:
                print(f"Skipping file {audio_path} due to error: {str(e)}")
                continue  # Skip failed MP3 conversions
        
        # Check if the file (converted WAV or original non-MP3) can be loaded
        try:
            waveform, sr = torchaudio.load(target_path)
            new_paths.append(target_path)
            valid_indices.append(idx)
        except Exception as e:
            print(f"Skipping file {target_path} due to error: {str(e)}")
            continue  # Skip files that fail to load
    
    # Update DataFrame with only valid files
    df = df.loc[valid_indices].copy()
    df['file_path'] = new_paths
    return df

# Convert MP3 files to WAV in DataFrames
final_train_df = convert_mp3_to_wav(final_train_df, "train")
final_val_df = convert_mp3_to_wav(final_val_df, "val")
final_test_df = convert_mp3_to_wav(final_test_df, "test")

Converting MP3 to WAV for train:   0%|          | 135/32000 [00:12<54:15,  9.79it/s]  

Skipping file /kaggle/input/the-fake-or-real-dataset/for-original/for-original/training/fake/file32972.mp3 due to error: Decoding failed. ffmpeg returned error code: 1

Output from ffmpeg/avlib:

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable

Converting MP3 to WAV for train:   7%|▋         | 2257/32000 [03:25<17:09, 28.88it/s]  

Skipping file /kaggle/input/the-fake-or-real-dataset/for-original/for-original/training/fake/file9875.mp3 due to error: Decoding failed. ffmpeg returned error code: 1

Output from ffmpeg/avlib:

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-

Converting MP3 to WAV for train:  11%|█         | 3501/32000 [05:15<58:25,  8.13it/s]  

...

Converting MP3 to WAV for train:  12%|█▏        | 3985/32000 [06:06<43:30, 10.73it/s]  

Skipping file /kaggle/input/the-fake-or-real-dataset/for-original/for-original/training/fake/file16643.mp3 due to error: Decoding failed. ffmpeg returned error code: 1

Output from ffmpeg/avlib:

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable

Converting MP3 to WAV for train:  18%|█▊        | 5630/32000 [08:33<29:15, 15.02it/s]  

Skipping file /kaggle/input/the-fake-or-real-dataset/for-original/for-original/training/fake/file27839.mp3 due to error: Decoding failed. ffmpeg returned error code: 1

Output from ffmpeg/avlib:

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable

Converting MP3 to WAV for test: 100%|██████████| 3200/3200 [01:16<00:00, 41.96it/s]


In [17]:
# Define target length
target_length = 80000  # 5 seconds at 16kHz
spectrogram_dir = '/kaggle/working/spectrograms/'
os.makedirs(spectrogram_dir, exist_ok=True)

# Function to process and save spectrograms
def preprocess_and_save_spectrograms(df, dataset_type):
    """
    Process all audio files in the dataset and save log-mel spectrograms as .pt files
    """
    spectrogram_paths = []
    valid_labels = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Preprocessing {dataset_type}"):
        audio_path = row['file_path']
        label = row['label_enc']
        spectrogram_path = os.path.join(spectrogram_dir, f"{dataset_type}_{idx}.pt")
        
        try:
            # Load audio file
            waveform, sr = torchaudio.load(audio_path)
            
            # Convert stereo to mono
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0)
            else:
                waveform = waveform.squeeze(0)
            
            # Trim or pad to target length
            if waveform.shape[0] > target_length:
                waveform = waveform[:target_length]
            elif waveform.shape[0] < target_length:
                waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[0]))
            
            # Convert to NumPy for ASTFeatureExtractor
            waveform_np = waveform.cpu().numpy()
            
            # Convert to log-mel spectrogram
            inputs = feature_extractor(
                waveform_np,
                sampling_rate=sample_rate,
                return_tensors="pt",
                padding=True
            )
            
            # Save spectrogram
            torch.save(inputs['input_values'], spectrogram_path)
            spectrogram_paths.append(spectrogram_path)
            valid_labels.append(label)
            
        except Exception as e:
            print(f"Skipping file {audio_path} due to error: {e}")
            continue
    
    return spectrogram_paths, valid_labels


# Function to prepare spectrogram inputs
def prepare_spectrogram_data(inputs):
    """
    Prepare inputs from preprocessed spectrograms
    """
    return {"input_values": inputs}

In [18]:
# Preprocess datasets
train_spectrogram_paths, train_labels = preprocess_and_save_spectrograms(final_train_df, "train")
val_spectrogram_paths, val_labels = preprocess_and_save_spectrograms(final_val_df, "val")
test_spectrogram_paths, test_labels = preprocess_and_save_spectrograms(final_test_df, "test")

# Create new DataFrames for spectrograms
train_spectrogram_df = pd.DataFrame({'path': train_spectrogram_paths, 'label': train_labels})
val_spectrogram_df = pd.DataFrame({'path': val_spectrogram_paths, 'label': val_labels})
test_spectrogram_df = pd.DataFrame({'path': test_spectrogram_paths, 'label': test_labels})

# Collate function for DataLoader
def spectrogram_collate_fn(batch):
    """
    Load spectrograms and labels for a batch
    """
    paths = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    
    batch_spectrograms = []
    for p in paths:
        spectrogram = torch.load(p)
        batch_spectrograms.append(spectrogram)
    
    inputs = torch.stack(batch_spectrograms).squeeze(1).to(device)
    labels = torch.tensor(labels, dtype=torch.long).to(device)
    
    return paths, inputs, labels

# Create DataLoaders with collate function
train_loader = torch.utils.data.DataLoader(
    list(zip(train_spectrogram_df['path'], train_spectrogram_df['label'])),
    batch_size=32,
    shuffle=True,
    collate_fn=spectrogram_collate_fn
)
val_loader = torch.utils.data.DataLoader(
    list(zip(val_spectrogram_df['path'], val_spectrogram_df['label'])),
    batch_size=32,
    shuffle=False,
    collate_fn=spectrogram_collate_fn
)
test_loader = torch.utils.data.DataLoader(
    list(zip(test_spectrogram_df['path'], test_spectrogram_df['label'])),
    batch_size=32,
    shuffle=False,
    collate_fn=spectrogram_collate_fn
)

Preprocessing test: 100%|██████████| 3200/3200 [00:52<00:00, 60.85it/s]


In [ ]:
# Training loop
best_val_acc = 0
best_model_state = None

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for batch in tqdm(train_loader):
        paths, inputs, labels = batch
        
        # Process spectrogram data
        inputs = prepare_spectrogram_data(inputs)
        
        # Forward pass
        outputs = model(**inputs).logits
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * labels.size(0)
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
    
    train_loss /= train_total
    train_acc = train_correct / train_total
    
    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    val_preds = []
    val_trues = []
    
    with torch.no_grad():
        for batch in val_loader:
            paths, inputs, labels = batch
            
            # Process spectrogram data
            inputs = prepare_spectrogram_data(inputs)
            
            outputs = model(**inputs).logits
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * labels.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            val_preds.extend(predicted.detach().cpu().numpy())
            val_trues.extend(labels.detach().cpu().numpy())
    
    val_loss /= val_total
    val_acc = val_correct / val_total
    
    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    print("\nValidation Classification Report:")
    print(classification_report(val_trues, val_preds, target_names=le.classes_))
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model.state_dict()
        print("✅ Best validation accuracy updated")

# Save best model
if best_model_state is not None:
    torch.save(best_model_state, '/kaggle/working/best_ast_model.pth')
    print("✅ Saved best model")

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)

# Evaluation on test set
model.eval()
test_loss = 0
test_correct = 0
test_total = 0
test_preds = []
test_trues = []

with torch.no_grad():
    for batch in test_loader:
        paths, inputs, labels = batch
        
        # Process spectrogram data
        inputs = prepare_spectrogram_data(inputs)
        
        outputs = model(**inputs).logits
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * labels.size(0)
        _, predicted = torch.max(outputs, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        test_preds.extend(predicted.detach().cpu().numpy())
        test_trues.extend(labels.detach().cpu().numpy())

test_loss /= test_total
test_acc = test_correct / test_total

print(f"\nTest Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")
print("\nTest Classification Report:")
print(classification_report(test_trues, test_preds, target_names=le.classes_))

# Plot confusion matrix
cm = confusion_matrix(test_trues, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")

  0%|          | 0/1000 [00:00<?, ?it/s]/tmp/ipykernel_31/4224107856.py:21: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.

100%|██████████| 1000/1000 [25:40<00:00,  1.54s/it]

Epoch 1/10: Train Loss: 0.4015, Train Acc: 0.8778, Val Loss: 0.1964, Val Acc: 0.9472

Validation Classification Report:
              precision    recall  f1-score   support

  scene_fake       0.97      0.95      0.96       800
  scene_real       0.95      0.98      0.97       800
  voice_fake       0.91      0.96      0.94       800
  voice_real       0.96      0.90      0.93       800

    accuracy                           0.95      3200
   macro avg       0.95      0.95      0.95      3200
weighted avg       0.95      0.95      0.95      3200

✅ Best validation accuracy updated


  0%|          | 0/1000 [00:00<?, ?it/s]/tmp/ipykernel_31/4224107856.py:21: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.

100%|██████████| 1000/1000 [25:09<00:00,  1.51s/it]

Epoch 2/10: Train Loss: 0.1583, Train Acc: 0.9553, Val Loss: 0.1259, Val Acc: 0.9684

Validation Classification Report:
              precision    recall  f1-score   support

  scene_fake       0.98      0.98      0.98       800
  scene_real       0.98      0.98      0.98       800
  voice_fake       0.95      0.97      0.96       800
  voice_real       0.97      0.94      0.95       800

    accuracy                           0.97      3200
   macro avg       0.97      0.97      0.97      3200
weighted avg       0.97      0.97      0.97      3200

✅ Best validation accuracy updated


  0%|          | 0/1000 [00:00<?, ?it/s]/tmp/ipykernel_31/4224107856.py:21: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.

100%|██████████| 1000/1000 [24:22<00:00,  1.46s/it]

...

  0%|          | 0/1000 [00:00<?, ?it/s]/tmp/ipykernel_31/4224107856.py:21: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.

100%|██████████| 1000/1000 [24:01<00:00,  1.44s/it]

Epoch 5/10: Train Loss: 0.0717, Train Acc: 0.9808, Val Loss: 0.0662, Val Acc: 0.9803

Validation Classification Report:
              precision    recall  f1-score   support

  scene_fake       0.99      0.99      0.99       800
  scene_real       0.99      0.99      0.99       800
  voice_fake       0.96      0.98      0.97       800
  voice_real       0.98      0.95      0.97       800

    accuracy                           0.98      3200
   macro avg       0.98      0.98      0.98      3200
weighted avg       0.98      0.98      0.98      3200

✅ Best validation accuracy updated


  0%|          | 0/1000 [00:00<?, ?it/s]/tmp/ipykernel_31/4224107856.py:21: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.

100%|██████████| 1000/1000 [24:00<00:00,  1.44s/it]

Epoch 6/10: Train Loss: 0.0621, Train Acc: 0.9832, Val Loss: 0.0619, Val Acc: 0.9844

Validation Classification Report:
              precision    recall  f1-score   support

  scene_fake       0.99      0.99      0.99       800
  scene_real       0.99      0.99      0.99       800
  voice_fake       0.99      0.97      0.98       800
  voice_real       0.97      0.98      0.98       800

    accuracy                           0.98      3200
   macro avg       0.98      0.98      0.98      3200
weighted avg       0.98      0.98      0.98      3200

✅ Best validation accuracy updated


  0%|          | 0/1000 [00:00<?, ?it/s]/tmp/ipykernel_31/4224107856.py:21: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.

 13%|█▎        | 134/1000 [03:13<20:49,  1.44s/it]